# 07 - Strict Time-Aware Split

Goal: evaluate the models under a stricter split than random sampling.

This notebook uses a class-aware time split: within each binary class, older traffic is used for training and newer traffic is used for validation/test. This keeps the binary evaluation balanced while still introducing temporal shift.

In [ ]:
from argparse import Namespace
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.create_strict_splits import create_strict_splits  # noqa: E402
from src.build_graph_arrays import build_graph_arrays  # noqa: E402

STRICT_DIR = PROJECT_ROOT / "data" / "strict_time_balanced"
STRICT_RESULTS = PROJECT_ROOT / "results" / "strict_time_balanced"

print("Project root:", PROJECT_ROOT)

## Create strict split

In [ ]:
split_args = Namespace(
    input_path=PROJECT_ROOT / "data" / "processed" / "iot23_binary_sample.csv",
    output_dir=STRICT_DIR,
    strategy="class_time",
    rows_per_class=50_000,
    random_state=42,
)

split_metadata = create_strict_splits(split_args)
split_metadata["splits"]

## Build graph arrays

In [ ]:
graph_metadata = build_graph_arrays(
    processed_dir=STRICT_DIR,
    output_dir=STRICT_DIR,
)

graph_metadata

## Inspect strict results

The training commands used for the current saved results:

```powershell
python -m src.train_gat --arrays-path data\strict_time_balanced\graph_arrays.npz --model-dir models\strict_time_balanced --results-dir results\strict_time_balanced --epochs 50 --hidden-channels 64 --heads 4 --early-stopping --patience 8 --selection-metric val_tuned_f1 --message-passing-edges train
python -m src.train_baselines --arrays-path data\strict_time_balanced\graph_arrays.npz --model-dir models\strict_time_balanced --results-dir results\strict_time_balanced
```

In [ ]:
comparison = pd.read_csv(STRICT_RESULTS / "model_comparison.csv")
comparison

In [ ]:
gat_metrics = json.loads((STRICT_RESULTS / "gat_metrics.json").read_text(encoding="utf-8"))
gat_metrics["selection_policy"], gat_metrics["final_metrics"]["test"]